# BurstML: maximum-likelihood analysis of diffusing two-state FRET bursts

FRET_burstML (T. Hoffmann et al.) fits the *photon sequences* of freely
diffusing molecules with a joint model of diffusion through the focus,
conformational kinetics and photon counting: the radial coordinate is
discretised, the combined evolution operator (diffusion + kinetics −
observation) is eigendecomposed once per parameter set, and each burst's
likelihood is a product of diagonal propagations in the eigenbasis. Brightness,
diffusion time, exchange rate, populations, FRET efficiencies and background
are all fitted at once, without binning the bursts.

`tttrlib.BurstML` is a std-only C++ port of the original MATLAB/MEX code
(``mlhDiffNTRbkg_MT.cpp``); its negative log-likelihood is identical to the
MEX to 1e-12 and 8× faster. This example simulates bursts from a two-state
model with the forward model the likelihood assumes, evaluates the likelihood
profile around the truth, and runs the Nelder–Mead fit.

Parameter layout for ``n`` states and 2 colours (5n values):
``n0[1..n]`` brightness (counts/ms), ``tau[1..n]`` diffusion times (ms),
``k`` (n−1 exchange rates, 1/ms), ``f`` (n−1 relative populations),
``E[1..n]`` FRET efficiencies, ``bkg[acceptor, donor]`` (counts/ms).


In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

import tttrlib

## Simulating bursts with BurstML's forward model
A molecule enters at a random radial position, its brightness follows the
3D-Gaussian profile exp(-2 q^2), it random-walks in q, may switch state, and
emits photons whose colour depends on the state's FRET efficiency; a burst
ends when the inter-photon time exceeds a threshold. Colours: 0 = acceptor,
1 = donor; times in ms.



In [ ]:
def simulate_bursts(n0=(150.0, 300.0), tau_diff=(0.8, 2.5), rate_sum=1.0, peq=(0.4, 0.6),
                    eff=(0.85, 0.45), bkg=(2.5, 0.8), n_bursts=70, min_photons=30,
                    max_interphoton_ms=0.3, seed=42):
    rng = np.random.default_rng(seed)
    n_states = len(n0)
    qmax, jmax = 4.0, 50
    dq = qmax / jmax
    qaxis = np.array([(i + 1) * dq - dq / 2 for i in range(jmax)])
    profile = np.exp(-2.0 * qaxis ** 2)
    times_all, colours_all, offsets = [], [], [0]
    for _ in range(n_bursts):
        state = rng.choice(n_states, p=peq)
        q_idx = rng.choice(jmax, p=profile / profile.sum())
        brightness = profile[q_idx]
        photons_t, photons_c, t = [], [], 0.0
        while True:
            if rng.random() < rate_sum * 0.01:
                state = 1 - state
            total_rate = n0[state] * brightness + bkg[0] + bkg[1]
            dt = rng.exponential(1.0 / total_rate)
            t += dt
            if dt > max_interphoton_ms:
                break
            p_acc = (eff[state] * n0[state] * brightness + bkg[0]) / total_rate
            photons_t.append(t)
            photons_c.append(0 if rng.random() < p_acc else 1)
            q_idx = max(0, min(jmax - 1, q_idx + rng.integers(-1, 2)))
            brightness = profile[q_idx]
        if len(photons_t) >= min_photons:
            times_all.extend(photons_t)
            colours_all.extend(photons_c)
            offsets.append(len(times_all))
    return np.array(times_all), np.array(colours_all, dtype=np.int32), np.array(offsets, dtype=np.int64)


truth = dict(n0=(150.0, 300.0), tau_diff=(0.8, 2.5), rate_sum=1.0, peq=(0.4, 0.6),
             eff=(0.85, 0.45), bkg=(2.5, 0.8))
times, colours, offsets = simulate_bursts(**truth)
print(f"{len(offsets) - 1} bursts, {times.size} photons")

## The likelihood



In [ ]:
ml = tttrlib.BurstML()
ml.set_burst_data(times.tolist(), colours.tolist(), offsets.tolist())
ml.set_n_states(2)
ml.set_n_colours(2)
ml.set_jmax(15)          # radial bins: coarser = faster; 15 is plenty for a 2-state fit
ml.set_qmax(4.0)
ml.set_t_th(0.3)         # inter-photon threshold that ends a burst (ms)
ml.set_n_th(30.0)        # minimum photons per burst

# n0(2), tau(2), k(1), f(1), E(2), bkg(2)
p_true = [150.0, 300.0, 0.8, 2.5, 1.0, 0.4 / 0.6, 0.85, 0.45, 2.5, 0.8]
nll_true = ml.neg_log_likelihood(p_true)
print(f"-log L at the simulation truth: {nll_true:.1f}")

A 1-D profile in each FRET efficiency, everything else held at the truth: the
minima sit at the simulated values.



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, idx, label in zip(axes, (6, 7), ("E of state 1", "E of state 2")):
    grid = np.linspace(0.2, 0.98, 40)
    prof = []
    for v in grid:
        p = list(p_true)
        p[idx] = v
        prof.append(ml.neg_log_likelihood(p))
    ax.plot(grid, np.asarray(prof) - nll_true)
    ax.axvline(p_true[idx], color="C3", ls="--", label="simulated")
    ax.set_xlabel(label)
    ax.set_ylabel(r"$-\log L$ - min")
    ax.legend()
fig.tight_layout()

A 2-D profile in the two brightnesses.



In [ ]:
n0_1 = np.linspace(80, 240, 15)
n0_2 = np.linspace(200, 420, 15)
Z = np.empty((n0_2.size, n0_1.size))
for i, b in enumerate(n0_2):
    for j, a in enumerate(n0_1):
        p = list(p_true)
        p[0], p[1] = a, b
        Z[i, j] = ml.neg_log_likelihood(p)
fig, ax = plt.subplots(figsize=(5.2, 4.2))
cs = ax.contourf(n0_1, n0_2, Z - Z.min(), levels=25, cmap="viridis")
ax.plot(p_true[0], p_true[1], "r+", ms=14, mew=2, label="simulated")
fig.colorbar(cs, label=r"$-\log L$ - min")
ax.set_xlabel("n0 state 1 / counts ms$^{-1}$")
ax.set_ylabel("n0 state 2 / counts ms$^{-1}$")
ax.legend()
fig.tight_layout()

## The fit: Nelder-Mead from a deliberately off initial guess, within bounds
The FRET efficiencies come back where they were simulated; brightness and
diffusion time are only weakly determined here -- the toy simulator above
moves the molecule by one radial bin per photon rather than diffusing it,
so its brightness/transit-time correlation is not the model's, and those two
parameters trade against each other along a likelihood valley. On real data
(or the simulator's own SimEngine transits) they separate.



In [ ]:
init = [200.0, 350.0, 1.0, 3.0, 0.8, 0.5, 0.8, 0.5, 3.0, 1.0]
lb = [50, 100, 0.1, 0.3, 0.1, 0.2, 0.7, 0.3, 0.5, 0.5]
ub = [500, 600, 3.0, 10.0, 10.0, 0.8, 0.95, 0.6, 5.0, 3.0]
t0 = time.perf_counter()
res = ml.fit(init, lb, ub, n_states=2, n_colours=2, jmax=15, qmax=4.0, t_th=0.3, n_th=30.0)
print(f"fit: {time.perf_counter() - t0:.1f} s, {res.iterations} iterations, status {res.status}, "
      f"logL {res.log_likelihood:.1f}, BIC {res.bic:.1f}")
names = ["n0[1]", "n0[2]", "tau[1]", "tau[2]", "k", "f", "E[1]", "E[2]", "bkg_A", "bkg_D"]
for n, pf, pt in zip(names, res.params, p_true):
    print(f"  {n:7s} fitted {pf:8.3f}   simulated {pt:8.3f}")
plt.show()